# 주주총회 소집공고 조회 테스트

OpenDART API를 통해 상장기업의 **주주총회 소집공고**를 검색하고, 의안별 상세 내용을 확인하는 노트북.

## 사용법

셀을 **위에서 아래 순서대로** 실행하면 됩니다.

| 단계 | 셀 | 설명 |
|------|-----|------|
| 0 | 초기화 | DartClient 로드 (최초 1회) |
| 1 | 기업 검색 | `QUERY`에 **종목코드**, **회사명**, 또는 **corp_code** 입력 → 소집공고 목록 반환 |
| 2 | 본문 가져오기 | `FILING_INDEX`로 공고 선택 → 의안 목록 자동 파싱 |
| 3 | 의안 상세 | `AGENDA_INDEX`로 의안 선택 → 해당 의안의 상세 내용 표시 |
| 4 | 전체 본문 | (선택) 전체 텍스트가 필요할 때 |

## 검색 예시

```python
QUERY = "033780"       # 종목코드 (KT&G)
QUERY = "삼성전자"      # 회사명
QUERY = "00126380"     # DART corp_code
```

> **참고**: DART에는 영문 브랜드명(KT&G)이 아닌 한글 정식명칭(케이티앤지)으로 등록되어 있습니다.  
> 회사명으로 검색이 안 되면 **종목코드**로 검색하세요.

In [1]:
import asyncio
import re

from open_proxy_mcp.dart.client import DartClient

client = DartClient()
print('DartClient 초기화 완료')

DartClient 초기화 완료


## 1. 소집공고 검색

종목코드, 회사명, 또는 corp_code로 검색 가능.

In [2]:
# === 여기에 검색할 기업 입력 ===
QUERY = "033780"  # 종목코드, 회사명, 또는 corp_code
# ================================

async def search_agm(query: str):
    # 기업 조회
    corp = await client.lookup_corp_code(query)
    if not corp:
        print(f"'{query}'에 해당하는 기업을 찾을 수 없습니다.")
        print("종목코드(예: 033780)로 다시 시도해보세요.")
        return None, None
    
    print(f"회사명: {corp['corp_name']}")
    print(f"종목코드: {corp['stock_code']}")
    print(f"corp_code: {corp['corp_code']}")
    print()
    
    # 소집공고 검색
    result = await client.search_filings_by_ticker(
        ticker=query,
        bgn_de="20260101",
        end_de="20261231",
        pblntf_ty="E",
    )
    
    filings = [
        item for item in result.get("list", [])
        if "소집" in item.get("report_nm", "")
    ]
    
    if not filings:
        print("주주총회 소집공고가 없습니다.")
        return corp, None
    
    print(f"소집공고 {len(filings)}건 발견:\n")
    for i, f in enumerate(filings):
        print(f"  [{i}] {f['report_nm']} | {f['rcept_dt']} | {f['rcept_no']}")
    
    return corp, filings

corp, filings = await search_agm(QUERY)

회사명: 케이티앤지
종목코드: 033780
corp_code: 00244455

소집공고 1건 발견:

  [0] 주주총회소집공고 | 20260225 | 20260225005779


## 2. 소집공고 본문 가져오기

위 결과에서 원하는 공고의 인덱스를 선택.

In [3]:
# === 위 결과에서 인덱스 선택 ===
FILING_INDEX = 0
# ================================

async def fetch_agm_document(filings, index: int):
    if not filings or index >= len(filings):
        print("유효한 인덱스를 선택하세요.")
        return None, None
    
    target = filings[index]
    print(f"가져오는 중: {target['report_nm']} ({target['rcept_no']})\n")
    
    doc = await client.get_document(target["rcept_no"])
    text = doc["text"]
    images = doc["images"]
    
    print(f"본문 길이: {len(text):,}자")
    if images:
        print(f"첨부 이미지: {images}")
    
    # 의안 목록 파싱
    agenda_items = parse_agenda(text)
    
    print(f"\n{'='*60}")
    print(f"의안 {len(agenda_items)}개 발견:\n")
    for i, item in enumerate(agenda_items):
        print(f"  [{i}] {item['title']}")
    
    return text, agenda_items

def parse_agenda(text: str) -> list[dict]:
    """본문에서 의안 목록 파싱"""
    items = []
    
    # 기본 정보 (일시, 장소)
    date_match = re.search(r'일\s*시\s*[:：]?\s*(.+?)(?=\d+\.\s*장|$)', text)
    place_match = re.search(r'장\s*소\s*[:：]?\s*(.+?)(?=\d+\.\s*회|$)', text)
    
    if date_match or place_match:
        basic_info = "주총 기본정보"
        if date_match:
            basic_info += f" | 일시: {date_match.group(1).strip()[:50]}"
        if place_match:
            basic_info += f" | 장소: {place_match.group(1).strip()[:80]}"
        items.append({"title": basic_info, "keyword": "일시"})
    
    # 제N호 의안 패턴 추출
    pattern = r'제(\d+[-\d]*)호[^:：]*[:：]?\s*(.+?)(?=제\d+[-\d]*호|□|■|$)'
    matches = re.findall(pattern, text[:5000])  # 앞부분 의안 목록에서만
    
    seen = set()
    for num, title in matches:
        title_clean = title.strip()[:100]
        key = f"제{num}호"
        if key not in seen and title_clean:
            seen.add(key)
            items.append({"title": f"{key}: {title_clean}", "keyword": key})
    
    return items

full_text, agenda_items = await fetch_agm_document(filings, FILING_INDEX)

가져오는 중: 주주총회소집공고 (20260225005779)

본문 길이: 49,428자
첨부 이미지: ['조직도.jpg', 'bsm_260225.jpg']

의안 14개 발견:

  [0] 주총 기본정보 | 일시: 2026년 3월 26일(목) 오전 10시 | 장소: 대전광역시 대덕구 벚꽃길 71 (주)케이티앤지 인재개발원 비전홀
  [1] 제1호: 제39기 재무제표 및 이익잉여금처분계산서 승인의 건 2)
  [2] 제2호: 정관 일부 변경의 건 ㆍ
  [3] 제2-1호: 목적사업 추가 ㆍ
  [4] 제2-2호: 전자주주총회 도입 ㆍ
  [5] 제2-3호: 사외이사 명칭 변경 ㆍ
  [6] 제2-4호: 집중투표 규정 정비 ㆍ
  [7] 제2-5호: 감사위원 분리선출 인원 확대 ㆍ
  [8] 제2-6호: 사내이사 및 경영임원 퇴직금지급규정 정비 ㆍ
  [9] 제2-7호: 자기주식 보유 또는 처분에 관한 규정 신설 ※
  [10] 제3호: 사외이사 노환용 선임의 건 5)
  [11] 제5호: 감사위원회 위원 노환용 선임의 건 6)
  [12] 제6호: 감사위원회 위원이 되는 사외이사 한승수 선임의 건 ※
  [13] 제8호: 자기주식보유처분계획서 승인의 건 ※


## 3. 의안 상세 보기

위 의안 목록에서 인덱스를 선택하면 해당 내용을 표시.

In [4]:
# === 위 의안 목록에서 인덱스 선택 ===
AGENDA_INDEX = 0
# ====================================

def show_agenda_detail(text: str, agenda_items: list[dict], index: int, context_chars: int = 3000):
    if not agenda_items or index >= len(agenda_items):
        print("유효한 인덱스를 선택하세요.")
        return
    
    item = agenda_items[index]
    keyword = item["keyword"]
    
    print(f"{'='*60}")
    print(f"  {item['title']}")
    print(f"{'='*60}\n")
    
    # 경영참고사항 섹션에서 해당 의안 찾기 (뒷부분에 상세 내용)
    # □ 또는 ■ + 제N호 패턴으로 찾기
    detail_pattern = f'[□■].*?{re.escape(keyword)}'
    matches = list(re.finditer(detail_pattern, text))
    
    if matches:
        # 마지막 매치가 보통 상세 내용 (앞부분은 목록)
        start = matches[-1].start()
        excerpt = text[start:start + context_chars]
        print(excerpt)
        if start + context_chars < len(text):
            print(f"\n... (더 보려면 context_chars를 늘리세요)")
    else:
        # 키워드로 직접 검색
        idx = text.find(keyword)
        if idx >= 0:
            # 좀 더 뒤에서 찾기 (상세 내용은 보통 뒷부분)
            idx2 = text.find(keyword, idx + len(keyword))
            pos = idx2 if idx2 >= 0 else idx
            excerpt = text[max(0, pos - 100):pos + context_chars]
            print(excerpt)
        else:
            print("해당 의안의 상세 내용을 찾을 수 없습니다.")

show_agenda_detail(full_text, agenda_items, AGENDA_INDEX)

  주총 기본정보 | 일시: 2026년 3월 26일(목) 오전 10시 | 장소: 대전광역시 대덕구 벚꽃길 71 (주)케이티앤지 인재개발원 비전홀

□ 재무제표의 승인 ■ 제1호 : 제39기 재무제표 및 이익잉여금처분계산서 승인의 건 가. 해당 사업연도의 영업상황의 개요 ※ 상기 'III. 경영참고사항' 중 '1. 사업의 개요'를 참조하시기 바랍니다. 나. 해당 사업연도의 대차대조표(재무상태표)ㆍ손익계산서(포괄손익계산서)ㆍ이익잉여금처분계산서(안) ※ 본 재무제표는 외부감사 결과를 반영하지 않은 것으로서 외부감사 결과 재무제표에 경미한 변경이 생긴 경우 대표이사는 외부감사인의 감사보고서 내용을 반영하여 수정된 재무제표를 주주총회에 제출할 수 있으며 상세 내역은 향후 '감사보고서 제출' 공시를 참조하시기 바랍니다. 1) 연결재무제표 ① 연결재무상태표 연 결 재 무 상 태 표 제 39(당) 기 2025년 12월 31일 현재 제 38(전) 기 2024년 12월 31일 현재 주식회사 케이티앤지와 그 종속기업 (단위: 백만원) 과 목 주석 제 39(당)기 제 38(전)기 자산 유동자산 현금및현금성자산 5, 29, 30 913,503 1,135,968 기타금융자산 5, 29, 30 303,277 463,317 당기손익-공정가치금융자산 6, 29, 30 61,431 244,941 매출채권및기타채권 7, 22, 28, 29 1,691,394 1,561,652 파생상품자산 29, 30 4,965 - 재고자산 8 3,284,784 3,101,313 환불자산등 18 792 6,161 미수담배소비세등 490,835 404,017 선급금 30 70,515 124,642 선급비용 163,051 131,094 당기법인세자산 26 398 3,205 매각예정자산 4, 33 41,298 - 유동자산 합계 7,026,243 7,176,310 비유동자산 장기기타금융자산 5, 29, 30 38,478 30,704 장기예치금 29, 30 1,676,086 1,705,504 장기당기손익-공정가치금융자산 

In [5]:
show_agenda_detail(full_text, agenda_items, AGENDA_INDEX)

  주총 기본정보 | 일시: 2026년 3월 26일(목) 오전 10시 | 장소: 대전광역시 대덕구 벚꽃길 71 (주)케이티앤지 인재개발원 비전홀

□ 재무제표의 승인 ■ 제1호 : 제39기 재무제표 및 이익잉여금처분계산서 승인의 건 가. 해당 사업연도의 영업상황의 개요 ※ 상기 'III. 경영참고사항' 중 '1. 사업의 개요'를 참조하시기 바랍니다. 나. 해당 사업연도의 대차대조표(재무상태표)ㆍ손익계산서(포괄손익계산서)ㆍ이익잉여금처분계산서(안) ※ 본 재무제표는 외부감사 결과를 반영하지 않은 것으로서 외부감사 결과 재무제표에 경미한 변경이 생긴 경우 대표이사는 외부감사인의 감사보고서 내용을 반영하여 수정된 재무제표를 주주총회에 제출할 수 있으며 상세 내역은 향후 '감사보고서 제출' 공시를 참조하시기 바랍니다. 1) 연결재무제표 ① 연결재무상태표 연 결 재 무 상 태 표 제 39(당) 기 2025년 12월 31일 현재 제 38(전) 기 2024년 12월 31일 현재 주식회사 케이티앤지와 그 종속기업 (단위: 백만원) 과 목 주석 제 39(당)기 제 38(전)기 자산 유동자산 현금및현금성자산 5, 29, 30 913,503 1,135,968 기타금융자산 5, 29, 30 303,277 463,317 당기손익-공정가치금융자산 6, 29, 30 61,431 244,941 매출채권및기타채권 7, 22, 28, 29 1,691,394 1,561,652 파생상품자산 29, 30 4,965 - 재고자산 8 3,284,784 3,101,313 환불자산등 18 792 6,161 미수담배소비세등 490,835 404,017 선급금 30 70,515 124,642 선급비용 163,051 131,094 당기법인세자산 26 398 3,205 매각예정자산 4, 33 41,298 - 유동자산 합계 7,026,243 7,176,310 비유동자산 장기기타금융자산 5, 29, 30 38,478 30,704 장기예치금 29, 30 1,676,086 1,705,504 장기당기손익-공정가치금융자산 

## 4. 전체 본문 (필요 시)

전체 본문이 필요하면 아래 셀 실행.

In [6]:
# 전체 본문 출력 (길 수 있음)
# print(full_text)